# NFL Player Stats Explorer — 2023 & 2024 Seasons

Exploratory analysis of NFL player stats using `nfl-data-py` and `pandas`. Data sourced from [nflverse](https://github.com/nflverse/nflverse-data).

**Sections:**
1. Setup & Data Load
2. Top QBs by Passing Yards (2024)
3. QB Efficiency - TD:INT Ratio
4. Top RBs by Rushing Yards (2024)
5. Year-over-Year RB Efficiency (2023 vs 2024)
6. Top WRs by Receiving Yards (2024)

## 1. Setup & Data Load

In [1]:
import nfl_data_py as nfl
import pandas as pd

# Load seasonal stats for 2023 and 2024
df = nfl.import_seasonal_data([2023, 2024])

# Load player metadata for names, positions, teams
players_raw = nfl.import_players()
players = players_raw[['gsis_id', 'display_name', 'position', 'latest_team']]

print(f'Rows: {df.shape[0]} | Columns: {df.shape[1]}')
df.head()

Rows: 1195 | Columns: 58


,player_id,season,season_type,completions,attempts,passing_yards,passing_tds,interceptions,sacks,sack_yards,...,yac_sh,wopr_y,ry_sh,rtd_sh,rfd_sh,rtdfd_sh,dom,w8dom,yptmpa,ppr_sh
0,00-0023459,2023,REG,0,1,0.0,0,0.0,1.0,10.0,...,0.000000,0.000000,0.000000,0.000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
1,00-0023459,2024,REG,368,584,3897.0,28,11.0,40.0,302.0,...,0.000000,0.000000,0.000000,0.000,0.000000,0.000000,0.000000,0.000000,0.000000,0.177032
2,00-0024243,2023,REG,0,0,0.0,0,0.0,0.0,0.0,...,0.069712,0.057342,0.027384,0.125,0.036364,0.047619,0.076192,0.046907,0.197279,0.028467
3,00-0024243,2024,REG,0,0,0.0,0,0.0,0.0,0.0,...,0.006757,0.059271,0.008850,0.000,0.071429,0.055556,0.004425,0.007080,0.068966,0.010541
4,00-0026158,2023,REG,123,204,1616.0,13,8.0,8.0,57.0,...,0.000000,0.000000,0.000000,0.000,0.000000,0.000000,0.000000,0.000000,0.000000,0.191464


## 2. Top QBs by Passing Yards (2024)

Filter to QBs with at least 100 pass attempts to exclude backups and spot starters.

In [2]:
qbs = df[(df['season'] == 2024) & (df['attempts'] > 100)].copy()
qbs = qbs[['player_id', 'passing_yards', 'passing_tds', 'interceptions', 'attempts', 'passing_epa']]
qbs = qbs.sort_values('passing_yards', ascending=False)
qbs = qbs.merge(players, left_on='player_id', right_on='gsis_id', how='left')

qbs[['display_name', 'latest_team', 'passing_yards', 'passing_tds', 'interceptions', 'attempts']].head(10)

,display_name,latest_team,passing_yards,passing_tds,interceptions,attempts
0,Joe Burrow,CIN,4918.0,43,9.0,652
1,Jared Goff,DET,4629.0,37,12.0,539
2,Baker Mayfield,TB,4500.0,41,16.0,570
3,Geno Smith,NYJ,4320.0,21,15.0,578
4,Sam Darnold,SEA,4319.0,35,12.0,545
5,Lamar Jackson,BAL,4172.0,41,4.0,474
6,Patrick Mahomes,KC,3928.0,26,11.0,581
7,Aaron Rodgers,PIT,3897.0,28,11.0,584
8,Justin Herbert,LAC,3870.0,23,3.0,504
9,Brock Purdy,SF,3864.0,20,12.0,455


## 3. QB Efficiency - TD:INT Ratio & EPA

Passing yards alone don't tell the whole story. TD:INT ratio and Expected Points Added (EPA) per attempt are better efficiency metrics.

- **TD:INT ratio** - touchdowns per interception thrown
- **EPA per attempt** - how many points each pass attempt added on average vs. expected

In [3]:
qbs['td_int_ratio'] = qbs['passing_tds'] / qbs['interceptions'].replace(0, 0.5)  # avoid divide by zero
qbs['epa_per_attempt'] = qbs['passing_epa'] / qbs['attempts']

efficiency = qbs[['display_name', 'latest_team', 'passing_tds', 'interceptions', 'td_int_ratio', 'epa_per_attempt']]
efficiency = efficiency.sort_values('epa_per_attempt', ascending=False)

efficiency.head(10)

,display_name,latest_team,passing_tds,interceptions,td_int_ratio,epa_per_attempt
5,Lamar Jackson,BAL,41,4.0,10.250000,0.367453
1,Jared Goff,DET,37,12.0,3.083333,0.310764
13,Josh Allen,BUF,28,6.0,4.666667,0.267195
20,Tua Tagovailoa,ATL,19,7.0,2.714286,0.212216
2,Baker Mayfield,TB,41,16.0,2.562500,0.203912
0,Joe Burrow,CIN,43,9.0,4.777778,0.179480
24,Derek Carr,NO,15,5.0,3.000000,0.173514
9,Brock Purdy,SF,20,12.0,1.666667,0.173032
18,Jordan Love,GB,25,11.0,2.272727,0.138775
44,Michael Penix Jr.,ATL,3,3.0,1.000000,0.138601


## 4. Top RBs by Rushing Yards (2024)

Filter to RBs with at least 50 carries. Calculate yards per carry as a derived efficiency metric.

In [4]:
rbs = df[(df['season'] == 2024) & (df['carries'] > 50)].copy()
rbs['yards_per_carry'] = rbs['rushing_yards'] / rbs['carries']
rbs = rbs[['player_id', 'carries', 'rushing_yards', 'rushing_tds', 'rushing_epa', 'yards_per_carry']]
rbs = rbs.sort_values('rushing_yards', ascending=False)
rbs = rbs.merge(players, left_on='player_id', right_on='gsis_id', how='left')

rbs[['display_name', 'latest_team', 'rushing_yards', 'rushing_tds', 'carries', 'yards_per_carry']].head(10)

,display_name,latest_team,rushing_yards,rushing_tds,carries,yards_per_carry
0,Saquon Barkley,PHI,2005.0,13,345,5.811594
1,Derrick Henry,BAL,1921.0,16,325,5.910769
2,Bijan Robinson,ATL,1456.0,14,304,4.789474
3,Jonathan Taylor,IND,1431.0,11,303,4.722772
4,Jahmyr Gibbs,DET,1412.0,16,250,5.648000
5,Josh Jacobs,GB,1329.0,15,301,4.415282
6,Kyren Williams,LA,1299.0,14,316,4.110759
7,Chuba Hubbard,CAR,1195.0,10,250,4.780000
8,Aaron Jones,MIN,1138.0,5,255,4.462745
9,Bucky Irving,TB,1122.0,8,207,5.420290


## 5. Year-over-Year RB Efficiency (2023 vs 2024)

Compare yards per carry for the top 10 rushers in 2024 against their 2023 numbers. Who improved? Who regressed?

In [5]:
rbs23 = df[(df['season'] == 2023) & (df['carries'] > 50)].copy()
rbs23['yards_per_carry'] = rbs23['rushing_yards'] / rbs23['carries']

rbs24 = df[(df['season'] == 2024) & (df['carries'] > 50)].copy()
rbs24['yards_per_carry'] = rbs24['rushing_yards'] / rbs24['carries']

# Get top 10 rushers in 2024 by yards
top_ids = rbs24.sort_values('rushing_yards', ascending=False).head(10)['player_id'].tolist()

# Merge 2024 and 2023 side by side
comparison = rbs24[rbs24['player_id'].isin(top_ids)].merge(
    rbs23, on='player_id', suffixes=('_2024', '_2023'), how='left'
)

comparison['ypc_improvement'] = comparison['yards_per_carry_2024'] - comparison['yards_per_carry_2023']
comparison = comparison.merge(players, left_on='player_id', right_on='gsis_id', how='left')

comparison[[
    'display_name', 'latest_team',
    'rushing_yards_2024', 'rushing_yards_2023',
    'yards_per_carry_2024', 'yards_per_carry_2023',
    'ypc_improvement'
]].sort_values('ypc_improvement', ascending=False).round(2)

,display_name,latest_team,rushing_yards_2024,rushing_yards_2023,yards_per_carry_2024,yards_per_carry_2023,ypc_improvement
2,Saquon Barkley,PHI,2005.0,962.0,5.81,3.89,1.92
0,Derrick Henry,BAL,1921.0,1167.0,5.91,4.17,1.74
5,Chuba Hubbard,CAR,1195.0,902.0,4.78,3.79,0.99
3,Josh Jacobs,GB,1329.0,805.0,4.42,3.45,0.96
8,Jahmyr Gibbs,DET,1412.0,945.0,5.65,5.19,0.46
4,Jonathan Taylor,IND,1431.0,741.0,4.72,4.38,0.34
7,Bijan Robinson,ATL,1456.0,976.0,4.79,4.56,0.23
1,Aaron Jones,MIN,1138.0,656.0,4.46,4.62,-0.16
6,Kyren Williams,LA,1299.0,1144.0,4.11,5.02,-0.91
9,Bucky Irving,TB,1122.0,NaN,5.42,NaN,NaN


## 6. Top WRs by Receiving Yards (2024)

Filter to receivers with at least 50 targets. Calculate yards per reception and yards per target as efficiency metrics.

In [6]:
wrs = df[(df['season'] == 2024) & (df['targets'] >= 50)].copy()
wrs['yards_per_reception'] = wrs['receiving_yards'] / wrs['receptions']
wrs['yards_per_target'] = wrs['receiving_yards'] / wrs['targets']
wrs['catch_rate'] = wrs['receptions'] / wrs['targets']

wrs = wrs[['player_id', 'targets', 'receptions', 'receiving_yards', 'receiving_tds',
           'yards_per_reception', 'yards_per_target', 'catch_rate']]
wrs = wrs.sort_values('receiving_yards', ascending=False)
wrs = wrs.merge(players, left_on='player_id', right_on='gsis_id', how='left')

wrs[['display_name', 'latest_team', 'receiving_yards', 'receiving_tds',
     'targets', 'receptions', 'yards_per_target', 'catch_rate']].head(10).round(2)

,display_name,latest_team,receiving_yards,receiving_tds,targets,receptions,yards_per_target,catch_rate
0,Ja'Marr Chase,CIN,1708.0,17,175,127,9.76,0.73
1,Justin Jefferson,MIN,1533.0,10,154,103,9.95,0.67
2,Brian Thomas Jr.,JAX,1282.0,10,133,87,9.64,0.65
3,Drake London,ATL,1271.0,9,158,100,8.04,0.63
4,Amon-Ra St. Brown,DET,1263.0,12,141,115,8.96,0.82
5,Jerry Jeudy,CLE,1229.0,4,145,90,8.48,0.62
6,Malik Nabers,NYG,1204.0,7,170,109,7.08,0.64
7,CeeDee Lamb,DAL,1194.0,6,152,101,7.86,0.66
8,Brock Bowers,LV,1194.0,5,153,112,7.80,0.73
9,Ladd McConkey,LAC,1149.0,7,112,82,10.26,0.73


In [7]:
## 7. Save to SQLite Database

import sqlite3
conn = sqlite3.connect('nfl_stats.db')

qbs[['display_name', 'latest_team', 'passing_yards', 'passing_tds', 'interceptions', 'attempts']].to_sql('qb_stats', conn, if_exists='replace',index=False)
rbs[['display_name', 'latest_team', 'rushing_yards', 'rushing_tds', 'carries', 'yards_per_carry']].to_sql('rb_stats', conn, if_exists='replace',index=False)
wrs[['display_name', 'latest_team', 'receiving_yards', 'receiving_tds', 'targets', 'receptions', 'yards_per_target', 'catch_rate']].to_sql('wr_stats',conn, if_exists='replace', index=False)

pd.read_sql('SELECT * FROM qb_stats ORDER BY passing_yards DESC LIMIT 10', conn)

,display_name,latest_team,passing_yards,passing_tds,interceptions,attempts
0,Joe Burrow,CIN,4918.0,43,9.0,652
1,Jared Goff,DET,4629.0,37,12.0,539
2,Baker Mayfield,TB,4500.0,41,16.0,570
3,Geno Smith,NYJ,4320.0,21,15.0,578
4,Sam Darnold,SEA,4319.0,35,12.0,545
5,Lamar Jackson,BAL,4172.0,41,4.0,474
6,Patrick Mahomes,KC,3928.0,26,11.0,581
7,Aaron Rodgers,PIT,3897.0,28,11.0,584
8,Justin Herbert,LAC,3870.0,23,3.0,504
9,Brock Purdy,SF,3864.0,20,12.0,455


In [8]:
conn.close()

### What This Section Did                                                                                                                                 
                                                                                                                                                            
Saved the three cleaned DataFrames to a local SQLite database (`nfl_stats.db`) using pandas `.to_sql()`. Then read the QB table back out using a standard 
SQL query to confirm the data landed correctly.                                                                                                           
                                                                                                                                                        
**The pipeline pattern:**                                                                                                                                 
- Ingest: nfl-data-py pulled raw seasonal stats
- Transform: pandas filtered, joined, and calculated derived metrics                                                                                      
- Store: SQLite persisted the results as queryable tables   
- Retrieve: SQL query read the data back out    

In [9]:
conn = sqlite3.connect('/Users/westonhyman/dev/nfl-explorer/nfl_stats.db')
pd.read_sql('SELECT * FROM qb_efficiency LIMIT 10', conn)

,display_name,latest_team,passing_yards,passing_tds,interceptions,attempts,td_int_ratio,yards_per_attempt
0,Lamar Jackson,BAL,4172.0,41,4.0,474,10.25,8.80
1,Jared Goff,DET,4629.0,37,12.0,539,3.08,8.59
2,Brock Purdy,SF,3864.0,20,12.0,455,1.67,8.49
3,Jalen Hurts,PHI,2903.0,18,5.0,361,3.60,8.04
4,Jordan Love,GB,3389.0,25,11.0,425,2.27,7.97
5,Sam Darnold,SEA,4319.0,35,12.0,545,2.92,7.92
6,Baker Mayfield,TB,4500.0,41,16.0,570,2.56,7.89
7,Kirk Cousins,ATL,3508.0,18,16.0,453,1.13,7.74
8,Josh Allen,BUF,3731.0,28,6.0,483,4.67,7.72
9,Derek Carr,NO,2145.0,15,5.0,279,3.00,7.69


In [10]:
pd.read_sql('SELECT * FROM rb_efficiency LIMIT 10', conn)

,display_name,latest_team,rushing_yards,rushing_tds,carries,yards_per_carry,td_rate
0,Saquon Barkley,PHI,2005.0,13,345,5.81,3.77
1,Derrick Henry,BAL,1921.0,16,325,5.91,4.92
2,Bijan Robinson,ATL,1456.0,14,304,4.79,4.61
3,Jonathan Taylor,IND,1431.0,11,303,4.72,3.63
4,Jahmyr Gibbs,DET,1412.0,16,250,5.65,6.40
5,Josh Jacobs,GB,1329.0,15,301,4.42,4.98
6,Kyren Williams,LA,1299.0,14,316,4.11,4.43
7,Chuba Hubbard,CAR,1195.0,10,250,4.78,4.00
8,Aaron Jones,MIN,1138.0,5,255,4.46,1.96
9,Bucky Irving,TB,1122.0,8,207,5.42,3.86
